In [1]:
import argparse
import os
from pathlib import Path
from omegaconf import OmegaConf
from itertools import chain
from toolz import pipe, identity
from toolz.curried import map as map_c

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torchdiffeq import odeint
from lightning.pytorch import seed_everything
from filterpy.kalman import UnscentedKalmanFilter, MerweScaledSigmaPoints
from sklearn.metrics import accuracy_score

import plotly.graph_objects as go
from rich.progress import track

from experiment.chaotic_systems.fields import LorenzField, RosslerField, ChuaField

In [2]:
SEED = 847
TRAJ_LEN = 100
dt = 1e-1
d = 3
NOISE_SIGMA = 1e-1
X0_SIGMA = 1.

In [3]:
config = OmegaConf.load("/Users/sem-k32/10_sem/n_ode/experiment/chaotic_systems/config.yaml")
seed_everything(SEED)

Seed set to 847


847

In [4]:
def np_field_adapter(torch_field: nn.Module):
    def field(x: np.ndarray, dt: float):
        x = torch.from_numpy(x)
        t_mesh = torch.tensor([0., dt])
        return odeint(torch_field, x, t_mesh)[-1].numpy()
    
    return field

In [5]:
t_mesh = torch.arange(TRAJ_LEN) * dt
for target_field_indx, target_field in enumerate(
    [LorenzField(), RosslerField(), ChuaField()]
):
    x0 = torch.randn(d) * X0_SIGMA
    target_traj = odeint(target_field, x0, t_mesh)[1:]
    target_traj += torch.randn_like(target_traj) * NOISE_SIGMA
    target_traj_np = target_traj.cpu().numpy()

    fig = go.Figure()
    fig.add_trace(
        go.Scatter3d(
            x=target_traj_np[:, 0],
            y=target_traj_np[:, 1],
            z=target_traj_np[:, 2], mode='lines+markers',
            name=f"target_{target_field_indx}"
        )
    )

    pred_fields = [LorenzField(), RosslerField(), ChuaField()]
    for pred_field_indx, pred_field in enumerate(pred_fields):
        points = MerweScaledSigmaPoints(d, alpha=.1, beta=2., kappa=-1)
        ukf = UnscentedKalmanFilter(
            d, d, dt,
            hx=identity, fx=np_field_adapter(pred_field), points=points
        )
        ukf.Q *= 1e-6
        ukf.R *= NOISE_SIGMA ** 2
        ukf.x = np.zeros((d, ))
        ukf.P *= X0_SIGMA ** 2
        
        mu, cov = ukf.batch_filter(target_traj)
        traj_smooth, _, _ = ukf.rts_smoother(mu, cov)

        fig.add_trace(
            go.Scatter3d(
                x=traj_smooth[:, 0],
                y=traj_smooth[:, 1],
                z=traj_smooth[:, 2], mode='lines+markers',
                name=f"pred_{pred_field_indx}"
            )
        )

    fig.update_layout(
        scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
        title=f"Predicting system {target_field_indx}"
    )
    fig.update_traces(marker=dict(size=3))
    fig.show()